# γ-range analysis over the W bank (set 1), V3

**New in V3** (everything from V2 below is unchanged):
- **Intracellular and intercellular frustration of every cell**, taken from the blocks of `J_sys = I_N ⊗ K + J_cell ⊗ (γW)` (W scaled by γ, as in the simulation):
  - *intracellular*: the cell's own diagonal block (its GRN `K`); fraction of its interactions `K[A,B] σ_A σ_B` that are negative;
  - *intercellular*: the off-diagonal `J_cell ⊗ γW` blocks between the cell and its 8 neighbours; fraction of the nonzero couplings that are negative. At γ = 0 there are no couplings, so it is 0 (as `intercellular_frustration_density` returns), and γ = 0 is left out of the intercellular plots.
  - Both were checked cell-by-cell against `intracellular_frustration_density` and `intercellular_frustration_density` (with scaled `J_sys`) on test files: identical.
- **Exact cell counts**: for every γ and GRN, the number of cells at each frustration value over all final tissues of all sims and Ws. Intracellular frustration is always k/8 (every GRN has 8 edges); intercellular is k/(8 neighbours × nonzero W entries). The counts are exact and take a few kB, so the violins and histograms use **every cell**.
- New plots: mean intracellular and intercellular frustration vs γ; violins of the cell frustration distribution per GRN for each γ; histograms.
- The cache has a new name (`..._V3.jld2`), so the V2 cache is not touched.

---


Same plots as `Gammarange_frustration_analysis_set1.ipynb`, rewritten so it can run over **all** Ws.

**What changed**
- The original loader kept every array of every (W, γ, GRN) in memory, about **90 MB per W** (E_selfish alone is 10 000 sweeps × 10 sims per group). 1000 Ws would need about 90 GB.
- `summarize_W_bank` reads one file at a time, keeps only the numbers the plots need (a few kB per W), and adds the per-sweep curves into running sums. Memory no longer grows with the number of Ws.
- Files are read in parallel on all Julia threads.
- Per-sim final frustration is recomputed with whole-number arithmetic (`fast_tissue_frustration`, bit-identical to `tissue_frustration`, ~4× faster).
- The summary is cached to `cache_file`. Re-running after more W files have arrived only reads the new files.
- GRN matrices and W are taken from each file's `meta/`, so the K-matrix cell is no longer needed.
- Added: **per-sim final frustration** (see the note in the frustration section) and the number of unique expression profiles.

All original quantities were checked against the original notebook code on test files: they are identical (final energy to about 1e-11, from summation order).

**Threads:** check `Threads.nthreads()` below. If it is 1, the notebook still works, just more slowly. To get more threads in Jupyter, install a threaded kernel once from a Julia session:
`using IJulia; installkernel("Julia 8 threads", env = Dict("JULIA_NUM_THREADS" => "8"))`, then choose that kernel. On OnDemand, request the same number of cores for the session.

In [ ]:
using Plots
using Random
using Statistics
using JLD2
using CodecZlib  

# Lattice, Model and tissue_frustration (used to recompute per-sim final frustration)
include("../../../Model/opti/functions/ising_core_opti.jl")
include("../functions/function_Gammarange_analysis_opti.jl")

println("Julia threads: ", Threads.nthreads())

In [ ]:
### ----------- data location ----------- ###
base_path  = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy/gamma_data_V3"
cache_file = "/home/twwoodw/my_folder/compute/data/gamma_study/summary_gammarange_set1_V3.jld2"

function make_filename(base_path, W_name, β_name, σ_name, n_sims, max_sweeps)
    return joinpath(base_path,
        "$(W_name)_$(β_name)_$(σ_name)_$(n_sims)_max$(max_sweeps)sweeps_flipequ.jld2")
end

### ----------- constants ----------- ###
n_cells    = 400
n_genes    = 9
n_sims     = 10
max_sweeps = 10_000

β_name = "betaInf"
σ_name = "σrand"

gammas      = [1.0, 0.8, 0.5, 0.3,0.25,0.22,0.20,0.18,0.16,0.14,0.125,0.12, 0.1, 0.08, 0.07, 0.06, 0.05, 0.04, 0.001, 0.0]
gamma_names = ["g1","g08","g05","g03","g025","g022","g02","g018","g016","g014","g0125","g012","g01","g008","g007","g006","g005","g004","g0001", "g0"]

grns = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]

### ----------- which Ws ----------- ###
W_range = 1:1000
W_signs = [1]        # [1] = W only (as before), [-1] = -W only, [1, -1] = both

# Run: summarise all Ws
### ONLY do once
The first run reads every file. Later runs load `cache_file` and only read files that are not in it yet. Missing files (e.g. jobs still running) are skipped.

If you change `gamma_names`, `grns`, `n_sims` or `max_sweeps`, the cache is rebuilt automatically. To force a full rebuild, delete `cache_file`.

In [ ]:
W_name(i, sgn) = sgn == 1 ? "W$(i)" : "-W$(i)"

all_files = [make_filename(base_path, W_name(i, s), β_name, σ_name, n_sims, max_sweeps)
             for i in W_range for s in W_signs]
files = filter(isfile, all_files)
println("$(length(files)) of $(length(all_files)) files exist")

@time S = summarize_W_bank(files;
    gamma_names = gamma_names,
    grns        = grns,
    n_cells     = n_cells,
    n_sims      = n_sims,
    max_sweeps  = max_sweeps,
    cache_file  = cache_file
);

# Read: summerize all Ws
### if summerize has been run for a set of gammas, Ws and GRNs juts load the file containing all computed metrics

In [ ]:
d = load(cache_file)
println("grns in cache: ", d["settings"]["grns"])

In [ ]:
# ── Parameters (from SLURM + run_ising_Wbank.jl) ─────────────────────────────

base_path  = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy/gamma_data_V3"
β_name     = "betaInf"
σ_name     = "σrand"
n_cells    = 400
n_sims     = 10
max_sweeps = 10000

cache_file = "/home/twwoodw/my_folder/compute/data/gamma_study/summary_gammarange_set1_V3.jld2"

# ── Load pre-computed summary ──────────────────────────────────────────────────
S = let d = load(cache_file)
    (; (Symbol(k) => v for (k, v) in d)...)
end

@assert S.settings["gamma_names"] == collect(gamma_names)  "gamma_names mismatch"
@assert S.settings["grns"]        == collect(grns)         "grns mismatch"
@assert S.settings["n_sims"]      == n_sims                "n_sims mismatch"
@assert S.settings["n_cells"]     == n_cells               "n_cells mismatch"
@assert S.settings["max_sweeps"]  == max_sweeps            "max_sweeps mismatch"

println("Loaded: $(length(S.W_files)) Ws | version: $(S.settings["summary_version"])")

In [ ]:
# what is in S (arrays are [sim, W, γ, GRN], [W, γ, GRN] or [sweep, γ, GRN])
n_W = length(S.W_files)
println("Ws in summary: ", n_W)
println("equilibrated: ", round(100 * mean(S.equilibrated), digits = 2), " % of all sims")
println("longest sim: ", maximum(S.sweeps_used), " sweeps")

# only W (not -W) when the summary contains both, e.g.
# S_pos = W_subset(S, n -> !startswith(n, "-"))

In [ ]:
gi(γ)  = findfirst(==(γ), gammas)        # γ value  -> index into the arrays
ki(Kn) = findfirst(==(Kn), grns)         # GRN name -> index
W_name(i, sgn) = sgn == 1 ? "W$(i)" : "-W$(i)"   # W index + sign -> file name, e.g. (2, 1) -> "W2"

grn_colors = Dict(
    "tree"       => :orange,
    "tri"        => :green,
    "negtri"     => :teal,
    "ICFF"       => :purple,
    "CFF"        => :violet,
    "negtritwin" => :blue,
    "negtritri"  => :cyan
)

# Per-sweep curves averaged over Ws
Same structure as before: `Wavg[K_name][γ]["sweeps" / "avg_flip_counts" / "avg_tissue_frustration"]`.

In [ ]:
Wavg = Dict()

for K_name in grns
    Wavg[K_name] = Dict()
    for γ in gammas
        sweeps, flips = W_average_curve(S, :flips, gi(γ), ki(K_name))
        _,      frus  = W_average_curve(S, :frus,  gi(γ), ki(K_name))

        Wavg[K_name][γ] = Dict(
            "sweeps"                 => sweeps,
            "avg_flip_counts"        => flips,
            "avg_tissue_frustration" => frus
        )
    end
end

# Final avg energy
## averaged over the n_sims and over the range of Ws

In [ ]:
final_E_avg = Dict()

for K_name in grns
    final_E_avg[K_name] = Dict()
    for γ in gammas
        # final energy of every sim of every W; every W has n_sims sims, so this equals
        # the original "average over sims, then over Ws"
        final_E_avg[K_name][γ] = mean(S.final_E[:, :, gi(γ), ki(K_name)])
    end
end

In [ ]:
gammas_plot = filter(x -> x ∉ (1.0, 0.8, 0.5), gammas)

γs_main  = filter(x -> x ∉ (0.0, 0.001), gammas_plot)
γs_break = [0.001, 0.0]

p_break = plot(
    xlabel    = "",
    ylabel    = "Final energy",
    title     = "",
    xticks    = ([0.001, 0.0], ["0.001", "0.0"]),
    xlims     = (-0.0005, 0.0015),
    xrotation = 45,
    legend    = false
)

p_main = plot(
    xlabel    = "γ",
    ylabel    = "",
    title     = "Final tissue energy vs γ",
    xticks    = 0.0:0.1:0.3,
    xrotation = 45
)

for K_name in grns
    y_main  = [final_E_avg[K_name][γ] for γ in γs_main]
    y_break = [final_E_avg[K_name][γ] for γ in γs_break]

    plot!(p_break, γs_break, y_break; marker = :circle, linewidth = 2, label = "", color = grn_colors[K_name])
    plot!(p_main,  γs_main,  y_main;  marker = :circle, linewidth = 2, label = K_name, color = grn_colors[K_name], dpi = 600)
end

p = plot(p_break, p_main,
    layout        = grid(1, 2, widths = [0.12, 0.88]),
    size          = (1000, 500),
    left_margin   = 8Plots.mm,
    bottom_margin = 10Plots.mm,
    dpi           = 600
)

display(p)
#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/final_tissue_energy_gammarange_$(σ_name)_set1_V6g0.png")

# Avg flip counts - should be zero as this is the equilibrium condition

In [ ]:
final_flips_avg = Dict()
final_flips_std = Dict()

for K_name in grns
    final_flips_avg[K_name] = Dict()
    final_flips_std[K_name] = Dict()
    for γ in gammas
        W_finals = S.last_flips[:, gi(γ), ki(K_name)]   # one value per W, as before

        final_flips_avg[K_name][γ] = mean(W_finals)
        final_flips_std[K_name][γ] = std(W_finals)
    end
end

In [ ]:
gammas_plot = filter(x -> x ∉ (1.0, 0.8, 0.5), gammas)  # includes 0.001 and 0.0

γs_main  = filter(x -> x ∉ (0.0, 0.001), gammas_plot)   # 0.06 … 0.3
γs_break = [0.001, 0.0]

p_break = plot(
    xlabel    = "",
    ylabel    = "Final avg flips per sweep",
    title     = "",
    xticks    = ([0.001, 0.0], ["0.001", "0.0"]),
    xlims     = (-0.0005, 0.0015),
    xrotation = 45,
    legend    = false
)

p_main = plot(
    xlabel    = "γ",
    ylabel    = "",
    title     = "Final flips vs γ ($(n_W) Ws)",
    xticks    = 0.0:0.1:0.3,
    xrotation = 45
)

for K_name in grns
    y_main  = [final_flips_avg[K_name][γ] for γ in γs_main]  ./ n_cells
    y_break = [final_flips_avg[K_name][γ] for γ in γs_break] ./ n_cells

    plot!(p_break, γs_break, y_break;
        marker    = :circle,
        linewidth = 2,
        label     = "",
        color     = grn_colors[K_name]
    )

    plot!(p_main, γs_main, y_main;
        marker    = :circle,
        linewidth = 2,
        label     = K_name,
        color     = grn_colors[K_name],
        dpi       = 600
    )
end

p = plot(p_break, p_main,
    layout        = grid(1, 2, widths = [0.12, 0.88]),
    size          = (1000, 500),
    left_margin   = 8Plots.mm,
    bottom_margin = 10Plots.mm,
    dpi           = 600
)

display(p)
#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/finaltissue_flips_gammarange_$(σ_name)_set1_V6g0.png")

# avg tissue frustration density

In [ ]:
final_frus_avg = Dict()

for K_name in grns
    final_frus_avg[K_name] = Dict()
    for γ in gammas
        final_frus_avg[K_name][γ] = mean(S.last_frus[:, gi(γ), ki(K_name)])    # average over Ws, as before
    end
end

In [ ]:
gammas_plot = filter(x -> x ∉ (1.0, 0.8, 0.5), gammas)

γs_main  = filter(x -> x ∉ (0.0, 0.001), gammas_plot)
γs_break = [0.001, 0.0]

p_break = plot(
    xlabel    = "",
    ylabel    = "Final tissue frustration density",
    title     = "",
    xticks    = ([0.001, 0.0], ["0.001", "0.0"]),
    xlims     = (-0.0005, 0.0015),
    xrotation = 45,
    legend    = false
)

p_main = plot(
    xlabel    = "γ",
    ylabel    = "",
    title     = "Final frustration vs γ ($(n_W) Ws)",
    xticks    = 0.0:0.1:0.3,
    xrotation = 45
)

for K_name in grns
    y_main  = [final_frus_avg[K_name][γ] for γ in γs_main]
    y_break = [final_frus_avg[K_name][γ] for γ in γs_break]

    plot!(p_break, γs_break, y_break; marker = :circle, linewidth = 2, label = "", color = grn_colors[K_name])
    plot!(p_main,  γs_main,  y_main;  marker = :circle, linewidth = 2, label = K_name, color = grn_colors[K_name])
end

p = plot(p_break, p_main,
    layout        = grid(1, 2, widths = [0.12, 0.88]),
    size          = (1000, 500),
    left_margin   = 8Plots.mm,
    bottom_margin = 10Plots.mm,
    dpi           = 600
)

display(p)
#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/finaltissue_frustration_gammarange_$(σ_name)_set1_V6g0.png")

## Note: the "last sweep" values come from the slowest sim only

`avg_tissue_frustration` and `avg_flip_counts` are saved as the mean over the sims **still running** at each sweep. A sim stops when it equilibrates, so at the last recorded sweep usually only the single slowest sim is left. The "final" values above (taken at the last non-NaN sweep, as in the original notebook) are therefore that one sim's value, not an average over the 10 sims.

The final tissues are saved, so the frustration of **every** sim's final tissue can be recomputed (`S.final_frus`). The check against the stored values matched exactly. The plot below uses that, averaged over sims and Ws, with ±1 std over Ws as a ribbon. Final flips per sim cannot be recovered from the saved data. Saving them would need a small change to the simulation script.

In [ ]:
gammas_plot = filter(!=(0.0), gammas)
x = 1:length(gammas_plot)

p = plot(
    xlabel    = "γ",
    ylabel    = "Final tissue frustration density",
    title     = "Final frustration vs γ",
    xticks    = (x, string.(gammas_plot)),
    xrotation = 45
)

for K_name in grns
    per_W = [vec(mean(S.final_frus[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in gammas_plot]  # mean over sims, per W
    y  = mean.(per_W)
    sd = n_W > 1 ? std.(per_W) : zeros(length(y))

    plot!(p, x, y;
        ribbon    = sd,
        fillalpha = 0.15,
        marker    = :circle,
        linewidth = 2,
        label     = K_name,
        color     = grn_colors[K_name],
        dpi       = 600
    )
end

display(p)
#savefig(p,  "~/my_folder/compute/data/gamma_study/gamma_plots_set1/finaltissue_frustration_allsims_gammarange_$(σ_name)_set1_V3extras.png")

# Number of unique gene expression profiles in the final tissue
Averaged over sims and Ws (±1 std over Ws).

In [ ]:
gammas_plot = filter(x -> x ∉ (1.0, 0.8, 0.5), gammas)

γs_main  = filter(x -> x ∉ (0.0, 0.001), gammas_plot)
γs_break = [0.001, 0.0]

p_break = plot(
    xlabel    = "",
    ylabel    = "Unique expression profiles in final tissue",
    title     = "",
    xticks    = ([0.001, 0.0], ["0.001", "0.0"]),
    xlims     = (-0.0005, 0.0015),
    xrotation = 45,
    legend    = false
)

p_main = plot(
    xlabel    = "γ",
    ylabel    = "",
    title     = "Cell types vs γ",
    xticks    = 0.0:0.1:0.3,
    xrotation = 45
)

for K_name in grns
    per_W_main  = [vec(mean(S.n_profiles[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs_main]
    per_W_break = [vec(mean(S.n_profiles[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs_break]
    y_main  = mean.(per_W_main);  sd_main  = n_W > 1 ? std.(per_W_main)  : zeros(length(y_main))
    y_break = mean.(per_W_break); sd_break = n_W > 1 ? std.(per_W_break) : zeros(length(y_break))

    plot!(p_break, γs_break, y_break; ribbon = sd_break, fillalpha = 0.15, marker = :circle, linewidth = 2, label = "", color = grn_colors[K_name])
    plot!(p_main,  γs_main,  y_main;  ribbon = sd_main,  fillalpha = 0.15, marker = :circle, linewidth = 2, label = K_name, color = grn_colors[K_name], dpi = 600)
end

p = plot(p_break, p_main,
    layout        = grid(1, 2, widths = [0.12, 0.88]),
    size          = (1000, 500),
    left_margin   = 8Plots.mm,
    bottom_margin = 10Plots.mm,
    dpi           = 600
)

display(p)
#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/n_profiles_gammarange_$(σ_name)_set1_V6g0.png")

## Mean intracellular and intercellular frustration vs γ
Tissue mean of each final tissue, averaged over sims and Ws (±1 std over the per-W means).

In [ ]:
gammas_plot = filter(x -> x ∉ (1.0, 0.8, 0.5, 0.001), gammas)

function frustration_vs_γ(field, γs, ylabel, title)
    p = plot(xlabel = "γ", ylabel = ylabel, title = title,
             xticks = 0.0:0.1:0.3, xrotation = 45, legend = :best)
    for K_name in grns
        per_W = [vec(mean(field[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs]
        y  = mean.(per_W)
        sd = n_W > 1 ? std.(per_W) : zeros(length(y))
        plot!(p, γs, y; ribbon = sd, fillalpha = 0.15, marker = :circle, linewidth = 2,
              label = K_name, color = grn_colors[K_name])
    end
    return p
end

p_loc = frustration_vs_γ(S.intracellular_frus, gammas_plot, "Mean intracellular frustration density",  "Intracellular frustration vs γ")
p_glo = frustration_vs_γ(S.final_frus, gammas_plot, "Mean intercellular frustration density", "Intercellular frustration vs γ")

p = plot(p_loc, p_glo, dpi = 600, layout = (1, 2), size = (1300, 500), left_margin = 5Plots.mm, bottom_margin = 8Plots.mm)
display(p)
#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intracellular_intercellular_frustration_vs_gamma_$(σ_name)_set1_V5g0.png")

In [ ]:
gammas_plot = filter(x -> x ∉ (1.0, 0.8, 0.5), gammas)

γs_main  = filter(x -> x ∉ (0.0, 0.001), gammas_plot)
γs_break = [0.001, 0.0]

function frustration_vs_γ(field, γs_main, γs_break, ylabel, title, fname)
    # collect all y values to set shared ylims
    all_y = Float64[]
    for K_name in grns
        per_W_main  = [vec(mean(field[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs_main]
        per_W_break = [vec(mean(field[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs_break]
        append!(all_y, mean.(per_W_main))
        append!(all_y, mean.(per_W_break))
    end
    ylims = (minimum(all_y) * 0.95, maximum(all_y) * 1.05)

    p_break = plot(
        xlabel    = "",
        ylabel    = ylabel,
        title     = "",
        xticks    = ([0.001, 0.0], ["0.001", "0.0"]),
        xlims     = (-0.0005, 0.0015),
        ylims     = ylims,
        xrotation = 45,
        legend    = false
    )

    p_main = plot(
        xlabel    = "γ",
        ylabel    = "",
        title     = title,
        xticks    = 0.0:0.1:0.3,
        ylims     = ylims,
        xrotation = 45,
        legend    = :best
    )

    for K_name in grns
        per_W_main  = [vec(mean(field[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs_main]
        per_W_break = [vec(mean(field[:, :, gi(γ), ki(K_name)]; dims = 1)) for γ in γs_break]
        y_main  = mean.(per_W_main);  sd_main  = n_W > 1 ? std.(per_W_main)  : zeros(length(y_main))
        y_break = mean.(per_W_break); sd_break = n_W > 1 ? std.(per_W_break) : zeros(length(y_break))

        plot!(p_break, γs_break, y_break; ribbon = sd_break, fillalpha = 0.15, marker = :circle, linewidth = 2, label = "",      color = grn_colors[K_name])
        plot!(p_main,  γs_main,  y_main;  ribbon = sd_main,  fillalpha = 0.15, marker = :circle, linewidth = 2, label = K_name, color = grn_colors[K_name])
    end

    p = plot(p_break, p_main,
        layout        = grid(1, 2, widths = [0.12, 0.88]),
        size          = (1000, 500),
        left_margin   = 8Plots.mm,
        bottom_margin = 10Plots.mm,
        dpi           = 600
    )
    display(p)
    savefig(p, fname)
end

frustration_vs_γ(S.intracellular_frus, γs_main, γs_break, "Mean intracellular frustration density",  "Intracellular frustration vs γ",
    "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intracellular_frustration_vs_gamma_$(σ_name)_set1_V6g0.png")

frustration_vs_γ(S.final_frus, γs_main, γs_break, "Mean intercellular frustration density", "Intercellular frustration vs γ",
    "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intercellular_frustration_vs_gamma_$(σ_name)_set1_V6g0.png")

# Intracellular and intercellular frustration (V3)
For each γ: every final tissue of every sim of every W. **intracellular** = the cell's own GRN block of `J_sys`; **intercellular** = the `J_cell ⊗ γW` blocks with its 8 neighbours.

- `S.intracellular_frus`, `S.final_frus`: tissue-mean intracellular / intercellular frustration per sim, `[sim, W, γ, GRN]`
- `frustration_distribution(S, :intracellular / :intercellular, gi(γ), ki(K_name))`: every frustration value and the average number of cells per tissue at that value (sums to `n_cells`), over all sims and Ws

In [ ]:
### ----------- plot settings ----------- ###
γ_show = 0.0          # γ for the histograms (the violins are drawn for every γ below)

bw_intracellular  = 0.02       # violin smoothing; intracellular values are k/8, so 0.02 keeps each value as its own bump
bw_intercellular = 0.004      # intercellular values are k/648 (≈ 0.0015 apart)
hist_bw_intercellular = 0.01  # bin width for the intercellular histograms

plot_gammas = filter(!=(0.0), gammas);   # no cell-cell couplings at γ = 0

## Histograms at `γ_show`
Average number of cells per tissue (out of `n_cells`) at each frustration value, over all sims and Ws. intracellular values are exact (k/8); intercellular values are grouped into bins of `hist_bw_intercellular`.

In [ ]:
g = gi(γ_show)

intracellular_plots = [begin
    v, c = frustration_distribution(S, :intracellular, g, k)
    μ = sum(v .* c) / sum(c)
    bar(v, c; bar_width = 0.1, color = grn_colors[K_name], legend = false,
        title = "$(K_name)  (μ=$(round(μ; digits = 2)))",
        xlabel = "Intracellular frustration", ylabel = "Cells per tissue", xlims = (-0.06, 1.06), ylims=(0,220))
end for (k, K_name) in enumerate(grns)]

p = plot(intracellular_plots..., layout = (2, 3), size = (900, 600), plot_title = "Intracellular frustration, γ = $(γ_show), $(n_W) Ws")
display(p)
#savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intracellular_frustration_hist_$(gamma_names[g])_$(σ_name)_set1_V3.png")

In [ ]:
g = gi(γ_show)

if γ_show == 0
    println("No cell-cell coupling at γ = 0: intercellular frustration is 0 for every cell.")
else
    intercellular_plots = [begin
        v, c = frustration_distribution(S, :intercellular, g, k)
        x, h = rebin(v, c, hist_bw_intercellular)
        keep = h .> 0
        bar(x[keep], h[keep]; bar_width = hist_bw_intercellular, color = grn_colors[K_name], legend = false, title = K_name, ylims=(0,35),
            xlabel = "Intercellular frustration", ylabel = "Cells per tissue")
    end for (k, K_name) in enumerate(grns)]

    p = plot(intercellular_plots..., layout = (2, 3), size = (900, 600), plot_title = "Intercellular frustration, γ = $(γ_show), $(n_W) Ws")
    display(p)
    #savefig(p, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intercellular_frustration_hist_$(gamma_names[g])_$(σ_name)_set1_V3extras.png")
end

# Violin plots 

In [ ]:
function frustration_violin_panels(results, tag; grns = grns, colors = grn_colors)
    function panel(field, ylabel, title; overlay)
        p = plot(xlabel = "GRN", ylabel = ylabel, title = title, legend = false)
        for (grn_idx, name) in enumerate(grns)
            r = results[name]
            if overlay
                for sim_entries in getfield(r, field)
                    violin!(p, [grn_idx], sim_entries, color = colors[name],
                            alpha = 0.5, linewidth = 1, linecolor = :black)
                end
            else
                vals = getfield(r, field)
                violin!(p, fill(grn_idx, length(vals)), vals, color = colors[name], linewidth = 0)
            end
        end
        xticks!(p, 1:length(grns), grns)
        return p
    end

    p1 = panel(:intracellular_sim_entries,  "intracellular cell frustration",          "intracellular cell frustration density";       overlay = true)
    p2 = panel(:intracellular_means,        "Mean intracellular cell frustration",     "Mean intracellular cell frustration density";  overlay = false)
    p3 = panel(:intercellular_sim_entries, "intercellular cell frustration density", "intercellular cell frustration density";      overlay = true)
    p4 = panel(:intercellular_means,       "Mean intercellular cell frustration",    "Mean intercellular cell frustration density"; overlay = false)

    return plot(p1, p2, p3, p4, layout = (2, 2), size = (1200, 950), dpi = 600,
                plot_title = tag, plot_titlefontsize = 16,
                titlefontsize = 12,
                left_margin = 5Plots.mm, bottom_margin = 5Plots.mm, top_margin = 3Plots.mm)
end

# W fixed

In [ ]:
using StatsPlots   # violin!

"""
cell_frustration_one_W(file, γ_name; grns, n_cells)

Per-cell INTRACELLULAR and INTERCELLULAR frustration of every sim's final tissue in one W file, at one γ.
Same definitions as summarize_W_file: intracellular = the cell's own K block of J_sys,
intercellular = the J_cell ⊗ γW blocks with its 8 neighbours.
Returns Dict GRN => (intracellular_sim_entries, intercellular_sim_entries, intracellular_means, intercellular_means).
"""
function cell_frustration_one_W(file, γ_name; grns, n_cells)
    lat = Lattice(n_cells; n_neighbors = 8)
    out = Dict{String, NamedTuple}()

    jldopen(file, "r") do f
        W    = f["meta/W"]
        γ    = Dict(zip(f["meta/gamma_names"], f["meta/gamma_values"]))[γ_name]
        Kmap = f["meta/K"]
        sW   = sign.(γ .* W)

        nsum = zeros(Int, size(W, 1), n_cells)
        lu = zeros(Int, n_cells)     # intracellular:  unsatisfied interactions per cell
        gu = zeros(Int, n_cells)     # intercellular: unsatisfied couplings per cell
        gn = zeros(Int, n_cells)     # intercellular: nonzero couplings per cell

        for Kn in grns
            σf = f["$(γ_name)/$(Kn)"]["sigma_finals_selfish"]
            Kp = grn_pairs(Kmap[Kn])
            nE = length(Kp)

            loc = Vector{Vector{Float64}}()
            glo = Vector{Vector{Float64}}()
            for sim in eachindex(σf)
                intracellular_cell_unsat!(lu, σf[sim], Kp)
                intercellular_cell_unsat!(gu, gn, σf[sim], sW, lat, nsum)
                push!(loc, nE == 0 ? zeros(n_cells) : lu ./ nE)
                push!(glo, [gn[c] == 0 ? 0.0 : gu[c] / gn[c] for c in 1:n_cells])
            end

            out[Kn] = (
                intracellular_sim_entries  = loc,
                intercellular_sim_entries = glo,
                intracellular_means        = mean.(loc),
                intercellular_means       = mean.(glo),
            )
        end
    end
    return out
end

In [ ]:
W_test_idx = 2
W_sign     = 1        # 1 → W, -1 → -W
γ_test     = 0.3     # intercellular frustration is 0 everywhere at γ = 0
test_sim   = 10

g_name = gamma_names[gi(γ_test)]
W_file = make_filename(base_path, W_name(W_test_idx, W_sign), β_name, σ_name, n_sims, max_sweeps)

frustration_results = cell_frustration_one_W(W_file, g_name; grns = grns, n_cells = n_cells)

# 20×20 grids of one sim (as in the original)
side = isqrt(n_cells)
intracellular_frus_grids  = Dict(K => reshape(frustration_results[K].intracellular_sim_entries[test_sim],  side, side) for K in grns)
intercellular_frus_grids = Dict(K => reshape(frustration_results[K].intercellular_sim_entries[test_sim], side, side) for K in grns)

# sanity check against the summary S (should be ~0)
w = findfirst(==(basename(W_file)), S.W_files)
if w !== nothing
    for K in grns
        dl = maximum(abs.(frustration_results[K].intracellular_means  .- S.intracellular_frus[:, w, gi(γ_test), ki(K)]))
        dg = maximum(abs.(frustration_results[K].intercellular_means .- S.final_frus[:, w, gi(γ_test), ki(K)]))
        println(rpad(K, 12), " max |Δ| intracellular = $dl, intercellular = $dg")
    end
end

final_plt = frustration_violin_panels(frustration_results, "$(W_name(W_test_idx, W_sign)), γ = $(γ_test)")
display(final_plt)
#savefig(final_plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/violin_frustration_$(W_name(W_test_idx, W_sign))_$(g_name)_$(σ_name)_set1_V3.png")


In [ ]:
W_test_idx = 2
W_sign     = 1        # 1 → W, -1 → -W
γ_test     = 0.05     # intercellular frustration is 0 everywhere at γ = 0
test_sim   = 10

g_name = gamma_names[gi(γ_test)]
W_file = make_filename(base_path, W_name(W_test_idx, W_sign), β_name, σ_name, n_sims, max_sweeps)

frustration_results = cell_frustration_one_W(W_file, g_name; grns = grns, n_cells = n_cells)

# 20×20 grids of one sim (as in the original)
side = isqrt(n_cells)
intracellular_frus_grids  = Dict(K => reshape(frustration_results[K].intracellular_sim_entries[test_sim],  side, side) for K in grns)
intercellular_frus_grids = Dict(K => reshape(frustration_results[K].intercellular_sim_entries[test_sim], side, side) for K in grns)

# sanity check against the summary S (should be ~0)
w = findfirst(==(basename(W_file)), S.W_files)
if w !== nothing
    for K in grns
        dl = maximum(abs.(frustration_results[K].intracellular_means  .- S.intracellular_frus[:, w, gi(γ_test), ki(K)]))
        dg = maximum(abs.(frustration_results[K].intercellular_means .- S.final_frus[:, w, gi(γ_test), ki(K)]))
        println(rpad(K, 12), " max |Δ| intracellular = $dl, intercellular = $dg")
    end
end

final_plt = frustration_violin_panels(frustration_results, "$(W_name(W_test_idx, W_sign)), γ = $(γ_test)")
display(final_plt)
#savefig(final_plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/violin_frustration_$(W_name(W_test_idx, W_sign))_$(g_name)_$(σ_name)_set1_V3.png")


# W pool results 

In [ ]:
W_range_plot = 1:20          # which Ws
W_sign_plot  = 1             # 1 → W, -1 → -W
γ_range      = 1.0
g_name_range = gamma_names[gi(γ_range)]

range_files = filter(isfile, [make_filename(base_path, W_name(i, W_sign_plot), β_name, σ_name, n_sims, max_sweeps)
                              for i in W_range_plot])
println("$(length(range_files)) of $(length(W_range_plot)) W files found")

per_W = Vector{Any}(undef, length(range_files))
Threads.@threads for i in eachindex(range_files)
    per_W[i] = cell_frustration_one_W(range_files[i], g_name_range; grns = grns, n_cells = n_cells)
end

# FIRST average over the sims of each W, THEN look at the spread over Ws
frustration_results_range = Dict{String, NamedTuple}()
for K in grns
    frustration_results_range[K] = (
        intracellular_sim_entries  = [reduce(vcat, r[K].intracellular_sim_entries)  for r in per_W],
        intercellular_sim_entries = [reduce(vcat, r[K].intercellular_sim_entries) for r in per_W],
        intracellular_means        = [mean(r[K].intracellular_means)  for r in per_W],
        intercellular_means       = [mean(r[K].intercellular_means) for r in per_W],
    )
end
per_W = nothing   # free memory


In [ ]:
tag = "W$(first(W_range_plot))–W$(last(W_range_plot)) ($(length(range_files)) Ws), γ = $(γ_range)"
range_plt = frustration_violin_panels(frustration_results_range, tag)
display(range_plt)
#savefig(range_plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/violin_frustration_Wrange$(first(W_range_plot))-$(last(W_range_plot))_$(g_name_range)_$(σ_name)_set1_V3.png")


In [ ]:
W_range_plot = 1:20          # which Ws
W_sign_plot  = 1             # 1 → W, -1 → -W
γ_range      = 0.05
g_name_range = gamma_names[gi(γ_range)]

range_files = filter(isfile, [make_filename(base_path, W_name(i, W_sign_plot), β_name, σ_name, n_sims, max_sweeps)
                              for i in W_range_plot])
println("$(length(range_files)) of $(length(W_range_plot)) W files found")

per_W = Vector{Any}(undef, length(range_files))
Threads.@threads for i in eachindex(range_files)
    per_W[i] = cell_frustration_one_W(range_files[i], g_name_range; grns = grns, n_cells = n_cells)
end

# FIRST average over the sims of each W, THEN look at the spread over Ws
frustration_results_range = Dict{String, NamedTuple}()
for K in grns
    frustration_results_range[K] = (
        intracellular_sim_entries  = [reduce(vcat, r[K].intracellular_sim_entries)  for r in per_W],
        intercellular_sim_entries = [reduce(vcat, r[K].intercellular_sim_entries) for r in per_W],
        intracellular_means        = [mean(r[K].intracellular_means)  for r in per_W],
        intercellular_means       = [mean(r[K].intercellular_means) for r in per_W],
    )
end
per_W = nothing   # free memory


In [ ]:
tag = "W$(first(W_range_plot))–W$(last(W_range_plot)) ($(length(range_files)) Ws), γ = $(γ_range)"
range_plt = frustration_violin_panels(frustration_results_range, tag)
display(range_plt)
#savefig(range_plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/violin_frustration_Wrange$(first(W_range_plot))-$(last(W_range_plot))_$(g_name_range)_$(σ_name)_set1_V3.png")
